# Section 0. Setup

In [3]:
import os
import gc
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils import class_weight
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
import whisper
from transformers import pipeline
import torch
from jiwer import wer, cer, process_words
from collections import defaultdict

# Reproducibility for Reviewer Validation
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

class Config:
    # Data Settings
    DATA_PATH = 'dataset_meld/'
    FILE_PREFIX = 'prepared_'
    SR = 16000
    MAX_DURATION = 3
    MAX_SAMPLES = SR * MAX_DURATION
    
    # Feature Settings
    N_MFCC = 40
    MAX_MFCC_LEN = 94  
    
    # Training Parameters 
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 30
    DROPOUT = 0.3
    
    DEFAULT_CLASSES = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise']

print("=== Hyperparameters ===")
for k, v in vars(Config).items():
    if not k.startswith('__'): print(f"{k}: {v}")

ModuleNotFoundError: No module named 'numpy'

In [2]:
import os
import shutil
import imageio_ffmpeg

# 1. Get the directory where imageio_ffmpeg stores its binary
ffmpeg_bin_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())

# 2. Whisper looks specifically for "ffmpeg.exe". Let's make sure a copy exists with that exact name.
target_exe = os.path.join(ffmpeg_bin_dir, "ffmpeg.exe")
if not os.path.exists(target_exe):
    shutil.copy(imageio_ffmpeg.get_ffmpeg_exe(), target_exe)

# 3. Inject this directory into the system PATH for this session
os.environ["PATH"] += os.pathsep + ffmpeg_bin_dir

# Section 1. ASR & Sentiment Classifier

### test with GPU

In [3]:
import torch
print(torch.__version__)

2.11.0+cu128


In [4]:
# 1. Check GPU availability and set up device formatting
cuda_available = torch.cuda.is_available()
whisper_device = "cuda" if cuda_available else "cpu"
hf_device = 0 if cuda_available else -1

print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"Activating GPU: {torch.cuda.get_device_name(0)}\n")
else:
    print("GPU not detected. Running on CPU instead.\n")

CUDA Available: True
Activating GPU: NVIDIA GeForce RTX 5080 Laptop GPU



In [5]:
# 2. Load Whisper ASR explicitly on the target device
print("Loading Whisper ASR...")
asr_model = whisper.load_model("base", device=whisper_device)

# 3. Load RoBERTa Sentiment Classifier on the target device
print("Loading RoBERTa Sentiment Classifier...")
sentiment_classifier = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    device=hf_device
)

print("\n✅ Both models loaded and pinned to GPU successfully!")

Loading Whisper ASR...
Loading RoBERTa Sentiment Classifier...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 44521.29it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Both models loaded and pinned to GPU successfully!


In [ ]:
AUDIO_DIRECTORY = "dataset_meld/audio/"
AUDIO_EXT = ".wav"
splits = ['train', 'dev', 'test']

os.makedirs("dataset_meld", exist_ok=True)

# --- GLOBAL STATS (per split) ---
stats = defaultdict(int)
failed_samples = []

# Define the text transformation pipeline for accurate WER comparison
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip()
])

# --- PROCESSING LOOP ---
for split in splits:

    print(f"\n================ {split.upper()} SPLIT ================\n")

    orig_csv = os.path.join("dataset_meld", f"{split}_sent_emo.csv")
    out_csv = os.path.join("dataset_meld", f"prepared_{split}_sent_emo.csv")

    processed_files = set()

    # Resume logic
    if os.path.exists(out_csv):
        try:
            existing_df = pd.read_csv(out_csv)
            if 'file_path' in existing_df.columns:
                processed_files = set(existing_df['file_path'].tolist())
                print(f"🔄 Resuming {split}: {len(processed_files)} already processed.")
        except:
            print(f"⚠️ Could not read {out_csv}, starting fresh.")

    print(f"Processing: {orig_csv}")

    df = pd.read_csv(orig_csv)

    # --- MOCK RUN MECHANISM ---
    MOCK_MODE = True  # Toggle this to False to run the full dataset
    if MOCK_MODE:
        # Take 10% of the data, sampled randomly
        df = df.sample(frac=0.10, random_state=SEED) 
        print(f"⚠️ MOCK MODE ACTIVE: Processing only 10% of {split} ({len(df)} samples)")
    # --------------------------

    prepared_data = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']

        raw_path = os.path.join(
            AUDIO_DIRECTORY,
            f"wav_{split}",
            f"dia{dia_id}_utt{utt_id}{AUDIO_EXT}"
        )

        file_path = os.path.abspath(raw_path)

        # ---------------------------
        # Duplicate check
        # ---------------------------
        if file_path in processed_files:
            stats["duplicate_sample"] += 1
            continue

        # ---------------------------
        # Missing audio
        # ---------------------------
        if not os.path.exists(file_path):
            stats["missing_audio"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "missing_audio"
            })
            continue

        # ---------------------------
        # Validate CSV fields
        # ---------------------------
        if pd.isna(row['Emotion']) or pd.isna(row['Sentiment']) or pd.isna(row['Utterance']):
            stats["invalid_csv_data"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "invalid_csv_data"
            })
            continue

        emo_gt = str(row['Emotion']).lower()
        sent_gt = str(row['Sentiment']).lower()
        trans_gt = str(row['Utterance']).strip()

        # ---------------------------
        # ASR transcription
        # ---------------------------
        try:
            with torch.no_grad():
                trans_asr = asr_model.transcribe(
                    file_path,
                    fp16=True,
                    language="en",
                    task="transcribe"
                )['text'].strip()

        except Exception as e:
            stats["asr_failed"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "asr_failed",
                "error": str(e)
            })
            continue

        # ---------------------------
        # Empty transcription check
        # ---------------------------
        if not trans_asr:
            stats["empty_transcription"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "empty_transcription"
            })
            continue

        # ---------------------------
        # Sentiment prediction
        # ---------------------------
        try:
            bert_out = sentiment_classifier(trans_asr)[0]
            sent_pred = bert_out['label'].lower()
            sent_score = bert_out['score']

        except Exception as e:
            stats["sentiment_failed"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "sentiment_failed",
                "error": str(e)
            })
            continue

        # ---------------------------
        # SUCCESS CASE
        # ---------------------------
        prepared_data.append({
            'emotion': emo_gt,
            'sentiment': sent_gt,  # MELD label (context-based ground truth)
            'textual_sentiment_predicted': sent_pred,
            'textual_sentiment_score': sent_score,
            'transcription_ground_truth': trans_gt,
            'transcription_ASR': trans_asr,
            'file_path': file_path
        })

        stats["success"] += 1

        # ---------------------------
        # Checkpoint saving
        # ---------------------------
        if len(prepared_data) >= 50:
            chunk_df = pd.DataFrame(prepared_data)
            chunk_df.to_csv(
                out_csv,
                mode='a',
                header=not os.path.exists(out_csv),
                index=False
            )
            prepared_data.clear()
            gc.collect()

    # Save remaining batch
    if prepared_data:
        chunk_df = pd.DataFrame(prepared_data)
        chunk_df.to_csv(
            out_csv,
            mode='a',
            header=not os.path.exists(out_csv),
            index=False
        )

    print(f"\n✅ Finished {split}. Saved to {out_csv}")

# ---------------------------
# FINAL REPORT
# ---------------------------
print("\n================ FINAL DATASET REPORT ================\n")

total_failures = (
    stats["missing_audio"] +
    stats["invalid_csv_data"] +
    stats["asr_failed"] +
    stats["empty_transcription"] +
    stats["sentiment_failed"]
)

print(f"Success samples           : {stats['success']}")
print(f"Duplicate samples         : {stats['duplicate_sample']}")
print(f"Missing audio             : {stats['missing_audio']}")
print(f"Invalid CSV data          : {stats['invalid_csv_data']}")
print(f"ASR failed                : {stats['asr_failed']}")
print(f"Empty transcription       : {stats['empty_transcription']}")
print(f"Sentiment prediction fail : {stats['sentiment_failed']}")

print(f"\nTOTAL FAILURES            : {total_failures}")

# ---------------------------
# SAVE FAILURE LOG
# ---------------------------
failed_df = pd.DataFrame(failed_samples)
failed_df.to_csv("dataset_meld/failed_samples_log.csv", index=False)

print(f"\n📁 Failure log saved: dataset_meld/failed_samples_log.csv")


================ TRAIN SPLIT ================

🔄 Resuming train: 8500 already processed.
Processing: dataset_meld\train_sent_emo.csv


100%|██████████| 9989/9989 [04:20<00:00, 38.41it/s] 



✅ Finished train. Saved to dataset_meld\prepared_train_sent_emo.csv

================ DEV SPLIT ================

Processing: dataset_meld\dev_sent_emo.csv


100%|██████████| 1109/1109 [03:20<00:00,  5.54it/s]



✅ Finished dev. Saved to dataset_meld\prepared_dev_sent_emo.csv

================ TEST SPLIT ================

Processing: dataset_meld\test_sent_emo.csv


100%|██████████| 2610/2610 [02:31<00:00, 17.22it/s] 


✅ Finished test. Saved to dataset_meld\prepared_test_sent_emo.csv

================ FINAL DATASET REPORT ================

Success samples           : 2556
Duplicate samples         : 8500
Missing audio             : 2
Invalid CSV data          : 0
ASR failed                : 2261
Empty transcription       : 388
Sentiment prediction fail : 1

TOTAL FAILURES            : 2652

📁 Failure log saved: dataset_meld/failed_samples_log.csv
